# تدريب كاشف اللوحات المصرية على Colab
# Train the Egyptian plate detector on Colab

هذا الدفتر يدرّب النموذج الذي يعطيك **المدى الأطول والسرعة الأعلى**.
الكاشف الكلاسيكي الحالي يعمل بدون تدريب لكنه يفقد اللوحات البعيدة.

This trains the model that gives you **longer range and higher speed**.
The classical fallback works untrained but loses distant plates.

---

## قبل أن تبدأ / before you start

**فعّل الـ GPU:** Runtime → Change runtime type → **T4 GPU** → Save

بدون GPU سيعمل التدريب لكنه سيستغرق ساعات بدل دقائق.

**الوقت المتوقع على T4:** ٢٠–٤٠ دقيقة لـ ١٥٠٠ صورة.

In [ ]:
# 1) تأكد من الـ GPU / confirm the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), \
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.'

## ٢ · البيانات / the data

**الأفضل بفارق كبير: صور من كاميرتك أنت.** بعد تشغيل النظام أسبوعًا،
اضغط مجلد `data/dataset/plates` وارفعه هنا.

*Nothing beats images from your own camera.* Public datasets are shot at
other angles, other lighting, other cameras — a model trained on them alone
will disappoint you at your gate.

اختر واحدًا من الخيارين في الخلية التالية.

In [ ]:
# 2) ارفع بياناتك / upload your data
#
#   الخيار أ: صورك أنت (الأفضل) — ارفع ملف zip فيه صور اللوحات
#   Option A: your own images (best) — upload a zip of plate photos
#
#   الخيار ب: مجموعة عامة من Roboflow — ضع مفتاحك وكود المشروع
#   Option B: a public Roboflow dataset — paste your key below

USE_MY_OWN_IMAGES = True

import os, zipfile, glob

if USE_MY_OWN_IMAGES:
    from google.colab import files
    print('اختر ملف zip فيه صورك / choose a zip of your images…')
    up = files.upload()
    for name in up:
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/raw')
    imgs = glob.glob('/content/raw/**/*.jpg', recursive=True) + \
           glob.glob('/content/raw/**/*.png', recursive=True)
    print(f'{len(imgs)} images ready')
    if len(imgs) < 150:
        print('\n⚠ أقل من ١٥٠ صورة — النتيجة ستكون ضعيفة.')
        print('⚠ Under 150 images will not train a usable detector.')
        print('  اجمع المزيد من كاميرتك، أو أضف مجموعة عامة أيضًا.')
else:
    !pip -q install roboflow
    from roboflow import Roboflow
    # ضع مفتاحك من roboflow.com ثم اختر مشروع لوحات مصرية
    rf = Roboflow(api_key='PUT_YOUR_KEY_HERE')
    project = rf.workspace('WORKSPACE').project('PROJECT')
    dataset = project.version(1).download('yolov8')
    print('downloaded to', dataset.location)

## ٣ · وسم الصور / labelling

الكاشف يحتاج **مربعًا حول كل لوحة**. إن كانت صورك غير موسومة، أسرع طريقة:

1. ارفعها على [roboflow.com](https://roboflow.com) (مجاني للمشاريع الصغيرة)
2. ارسم مربعًا حول اللوحة في كل صورة — صنف واحد فقط اسمه `plate`
3. صدّرها بصيغة **YOLOv8** وارجع للخيار ب أعلاه

وسم ٣٠٠ صورة يستغرق حوالي ساعة. هذه الساعة هي الفرق بين نظام يقرأ من
٣ أمتار ونظام يقرأ من ٨.

In [ ]:
# 3) درّب / train
from ultralytics import YOLO

DATA_YAML = '/content/datasets/data.yaml'   # عدّله حسب مكان بياناتك

model = YOLO('yolo11n.pt')      # nano: الأنسب لجهاز Jetson

model.train(
    data=DATA_YAML,
    epochs=120,
    imgsz=640,
    batch=16,
    patience=25,
    seed=0,
    # اللوحات مستطيلات صلبة: لا قلب رأسي، دوران بسيط فقط.
    # Plates are rigid rectangles - no vertical flips, mild rotation only.
    fliplr=0.0, flipud=0.0, degrees=4.0, shear=2.0,
    perspective=0.0004, hsv_v=0.5, close_mosaic=10,
    project='/content/runs', name='plate_detector', exist_ok=True,
)

In [ ]:
# 4) قيّم النتيجة قبل استخدامها / judge it before trusting it
m = YOLO('/content/runs/plate_detector/weights/best.pt')
metrics = m.val()
print(f'\nmAP50    : {metrics.box.map50:.3f}   <- اقرأ هذا الرقم')
print(f'mAP50-95 : {metrics.box.map:.3f}')
print(f'precision: {metrics.box.mp:.3f}')
print(f'recall   : {metrics.box.mr:.3f}   <- كم لوحة يجدها فعلاً')

if metrics.box.map50 < 0.80:
    print('\n⚠ أقل من ٠.٨٠ — لا تنشره بعد. تحتاج صورًا أكثر أو وسمًا أدق.')
    print('⚠ Under 0.80 mAP50: do not deploy yet. More or better-labelled data.')
else:
    print('\n✓ جاهز للنشر / good enough to deploy')

In [ ]:
# 5) نزّل النموذج / download the model
from google.colab import files
files.download('/content/runs/plate_detector/weights/best.pt')

print('''
الخطوة الأخيرة على جهازك:

  1. غيّر اسم الملف إلى:  plate_detector.pt
  2. ضعه في مجلد:         gate-anpr\\gate-anpr\\models\\
  3. أعد تشغيل النظام (START-GATE-ANPR)

سيلتقطه النظام تلقائيًا. للتأكد شغّل tools\\check-system.bat —
يجب أن يقول:  plate detector: yolo

On the Jetson, also export to TensorRT for 3-4x more speed:
  python tools/train_char_detector.py --export models/plate_detector.pt
''')

---

## ما الذي يتحسّن فعلاً / what actually improves

| | الكاشف الكلاسيكي | نموذج مدرَّب |
|---|---|---|
| المدى | ٢–٤ متر | ٦–١٠ متر بنفس الكاميرا |
| الليل والمطر | ضعيف | جيد |
| السرعة | متوسطة | أسرع (كشف واحد بدل بحث بالكنتور) |
| لوحات متسخة | يفشل | يتحمّل |

**هذا يدرّب الكاشف فقط** — الذي يجد اللوحة في الصورة. القارئ (الذي يحوّل
الصورة إلى نص) ما زال easyocr. لتدريب القارئ أيضًا استخدم
`tools/train_char_detector.py --task chars` بنفس الطريقة، لكن ابدأ بالكاشف:
مكسبه أكبر وبياناته أسهل.

---

# الجزء ٢: تدريب قارئ الحروف — هذا هو المهم
# Part 2: train the CHARACTER reader — this is the one that matters

**قِسْت easyocr على لوحات مصرية وكانت النتيجة:**

| | الدقة المقاسة |
|---|---|
| الحروف العربية | ~٥٠٪ في أفضل حالة |
| الأرقام | ~١٥٪ |

هذه أرقام غير صالحة لبوابة. **easyocr ليس حلاً** — هو مجرد شيء يعمل
بدون تدريب. القارئ المدرَّب هو الطريق الوحيد للمدى والسرعة معًا.

*I measured easyocr on Egyptian plates: ~50% on letters, ~15% on digits.
That is not usable at a gate. A trained character detector is the only
path to both range and speed.*

الخلايا التالية تولّد بيانات تدريب صناعية **بلوحات صغيرة ومشوّشة ومائلة**
عمدًا — لأن هذا بالضبط ما ستراه الكاميرا من ٥ أمتار. النموذج المدرَّب على
لوحات كبيرة نظيفة فقط يفشل على المدى البعيد.

In [ ]:
# توليد بيانات تدريب الحروف / generate character training data
!git clone -q https://github.com/YOUR/gate-anpr.git 2>/dev/null || true
# أو ارفع مجلد المشروع مضغوطًا واستخرجه هنا:
# from google.colab import files; files.upload()

%cd /content/gate-anpr
!pip -q install pillow fonttools opencv-python-headless pyyaml

# لوحات مُتعبة عمدًا: صغيرة، ضبابية، مائلة، إضاءة سيئة
!python tools/make_test_plates.py --count 4000 --clean --labels --degrade \
        --out /content/chars --seed 1

import glob, random, shutil, os, pathlib
imgs = sorted(glob.glob('/content/chars/images/*.jpg'))
random.seed(0); random.shuffle(imgs)
split = int(len(imgs) * 0.85)
for name, group in (('train', imgs[:split]), ('val', imgs[split:])):
    for sub in ('images', 'labels'):
        os.makedirs(f'/content/ds/{sub}/{name}', exist_ok=True)
    for i in group:
        stem = pathlib.Path(i).stem
        shutil.copy(i, f'/content/ds/images/{name}/{stem}.jpg')
        shutil.copy(f'/content/chars/labels/{stem}.txt',
                    f'/content/ds/labels/{name}/{stem}.txt')

classes = open('/content/chars/classes.txt').read().split('\n')
with open('/content/ds/chars.yaml', 'w') as f:
    f.write('path: /content/ds\ntrain: images/train\nval: images/val\nnames:\n')
    for i, c in enumerate(classes):
        f.write(f'  {i}: "{c}"\n')
print(f'{len(imgs)} plates, {len(classes)} character classes')

In [ ]:
# تحقّق من الوسم بعينك قبل التدريب / SEE the labels before training on them
# وسم خاطئ يدرّب نموذجًا خاطئًا بصمت — هذه الخلية تكشفه في ثانية.
import cv2, glob, pathlib
from google.colab.patches import cv2_imshow
import numpy as np

tiles = []
for ip in sorted(glob.glob('/content/ds/images/train/*.jpg'))[:6]:
    img = cv2.imread(ip); H, W = img.shape[:2]
    lp = ip.replace('/images/', '/labels/').replace('.jpg', '.txt')
    for line in open(lp).read().strip().split('\n'):
        _, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        cv2.rectangle(img, (int(cx-bw/2), int(cy-bh/2)),
                      (int(cx+bw/2), int(cy+bh/2)), (0, 0, 255), 2)
    tiles.append(cv2.resize(img, (350, 170)))
cv2_imshow(np.vstack([np.hstack(tiles[:3]), np.hstack(tiles[3:])]))
print('كل مربع يجب أن يحيط بحرف واحد بالضبط، بلا تداخل.')
print('Each box must hug exactly one character, with no overlap.')

In [ ]:
# درّب قارئ الحروف / train the character reader
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data='/content/ds/chars.yaml',
    epochs=80,
    imgsz=320,          # صغير عمدًا: الحروف صغيرة، والاستدلال يصير أسرع
    batch=32,
    patience=20, seed=0,
    # الحروف ليست متماثلة: القلب يحوّل حرفًا إلى حرف آخر
    fliplr=0.0, flipud=0.0,
    degrees=5.0, shear=3.0, perspective=0.0005,
    scale=0.5,          # يتعلّم القراءة على أحجام مختلفة = مدى أطول
    hsv_v=0.5, close_mosaic=10,
    project='/content/runs', name='char_detector', exist_ok=True,
)

m = YOLO('/content/runs/char_detector/weights/best.pt')
mt = m.val()
print(f'\nmAP50 {mt.box.map50:.3f}  precision {mt.box.mp:.3f}  recall {mt.box.mr:.3f}')
print('لا تنشره تحت ٠.٩٠ — قراءة الحروف تحتاج دقة أعلى من كشف اللوحة.'
      if mt.box.map50 < 0.90 else '✓ جاهز')

In [ ]:
from google.colab import files
files.download('/content/runs/char_detector/weights/best.pt')
print('''
على جهازك:
  1. غيّر الاسم إلى:  char_detector.pt
  2. ضعه في:          gate-anpr\\gate-anpr\\models\\
  3. أعد التشغيل

check-system.bat يجب أن يقول:  plate reader: yolo_chars

هذا النموذج أسرع من easyocr بعشرات المرات، فسيرفع النظام
عدد القراءات لكل إطار تلقائيًا (الميزانية تقيس نفسها).
''')